# M7: Clustering with K-Means and DBSCAN

**Course:** Introduction to Machine Learning  
**Instructor:** Becky Deitenbeck  
**Student:** Hannah Johnson  
**Date:** March 3, 2026  

---

## Overview

In this notebook, I apply **unsupervised learning** techniques to the Wine dataset:

- **Part A:** Data loading, exploration, and visualization
- **Part B:** K-Means clustering — finding the optimal K, plotting clusters, comparing to true labels
- **Part C:** DBSCAN clustering — experimenting with `eps` and `min_samples`, identifying noise points
- **Part D:** Reflection on real-world applications and ethical considerations

**Dataset:** Wine (sklearn) — 178 samples, 13 chemical features, 3 known cultivar classes  
**Note:** True labels are used *only* for evaluation (adjusted rand index), not during training — this is unsupervised learning.

---
## Part A: Data Exploration and Visualization

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully.')

### A.1 Load the Dataset

The **Wine dataset** from scikit-learn contains 178 samples of wine analyzed by their chemical composition across 13 features. There are 3 cultivar classes — though we will treat this as an unsupervised problem and only use the labels for evaluation.

In [ ]:
# Load Wine dataset
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['true_label'] = wine.target

print(f'Dataset shape: {df.shape}')
print(f'Features: {wine.feature_names}')
print(f'\nClass distribution:\n{df["true_label"].value_counts().sort_index()}')
df.head()

### A.2 Summary Statistics

Note that the features have very different scales (e.g., `proline` ranges into the hundreds while `nonflavanoid_phenols` is below 1). This confirms that **feature scaling is essential** before applying distance-based clustering algorithms.

In [ ]:
df.drop(columns=['true_label']).describe().round(2)

### A.3 Feature Distributions

Visualizing distributions helps identify skewed features, potential outliers, and natural separations between classes.

In [ ]:
# Histogram grid for all 13 features
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(wine.feature_names):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')

# Hide the 2 unused subplots
for j in range(len(wine.feature_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Wine Dataset — Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of 4 key features colored by true class
palette = {0: '#4C72B0', 1: '#DD8452', 2: '#55A868'}
plot_cols = ['alcohol', 'flavanoids', 'color_intensity', 'proline', 'true_label']
sns.pairplot(df[plot_cols], hue='true_label', palette=palette, diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot of Key Wine Features (colored by true cultivar class)', y=1.02, fontsize=13, fontweight='bold')
plt.show()

**Observations:**
- `flavanoids`, `alcohol`, and `proline` show the clearest visual separation between the three cultivar classes.
- `color_intensity` and `alcohol` are positively correlated.
- Several features are approximately normally distributed; `proline` is right-skewed.
- The three classes appear reasonably separable in 2D projections — K-Means should perform well.

---
## Part B: K-Means Clustering

### B.1 Feature Scaling

K-Means uses Euclidean distance, so features with larger ranges would dominate the clustering. `StandardScaler` transforms each feature to have mean=0 and standard deviation=1.

In [ ]:
X = df.drop(columns=['true_label']).values
y_true = df['true_label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaling complete.')
print(f'Mean after scaling (should be ~0): {X_scaled.mean(axis=0).round(2)}')

### B.2 Elbow Method — Finding the Optimal K

The **Elbow Method** plots inertia (within-cluster sum of squares) for K=1 to 8. The optimal K is where the curve bends — adding more clusters yields diminishing returns.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K=3 (elbow)')
axes[0].set_title('Elbow Method — Inertia vs. K', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# Silhouette scores
axes[1].plot(list(k_range), silhouettes, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K=3')
axes[1].set_title('Silhouette Score vs. K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Silhouette scores by K: {dict(zip(k_range, [round(s,3) for s in silhouettes]))}')

**Interpretation:** Both the Elbow plot and Silhouette score confirm **K=3** as the optimal number of clusters, which aligns with the 3 known wine cultivars. Beyond K=3, improvement diminishes significantly.

### B.3 Apply K-Means with K=3

We reduce to 2D using PCA for visualization, while the actual clustering is done in the full 13-dimensional scaled space.

In [ ]:
# Fit K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

# PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_.round(3)} '
      f'(total: {pca.explained_variance_ratio_.sum():.1%})')

# Adjusted Rand Index — how well clusters match true labels
ari = adjusted_rand_score(y_true, km_labels)
sil = silhouette_score(X_scaled, km_labels)
print(f'\nK-Means (K=3) Results:')
print(f'  Adjusted Rand Index : {ari:.4f}  (1.0 = perfect match)')
print(f'  Silhouette Score    : {sil:.4f}  (higher is better)')

In [ ]:
# Visualize K-Means clusters vs. true labels side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_km = ['#4C72B0', '#DD8452', '#55A868']

# K-Means clusters
for cluster in range(3):
    mask = km_labels == cluster
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_km[cluster], label=f'Cluster {cluster}',
                    s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
# Plot centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroids')
axes[0].set_title('K-Means Clusters (K=3) — PCA Projection', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].legend()

# True labels
for label in range(3):
    mask = y_true == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_km[label], label=f'Cultivar {label}',
                    s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
axes[1].set_title('True Cultivar Labels — PCA Projection', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[1].legend()

plt.suptitle(f'K-Means vs. True Labels  |  ARI = {ari:.3f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### B.4 Confusion Matrix — K-Means vs. True Labels

In [ ]:
# Note: K-Means cluster IDs may not match true label IDs — we remap for visualization
from scipy.optimize import linear_sum_assignment

def remap_labels(true_labels, pred_labels):
    """Remap cluster labels to best match true labels using Hungarian algorithm."""
    cm = confusion_matrix(true_labels, pred_labels)
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    return np.array([mapping[l] for l in pred_labels])

km_labels_remapped = remap_labels(y_true, km_labels)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_true, km_labels_remapped)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Cultivar 0', 'Cultivar 1', 'Cultivar 2'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('K-Means Cluster Assignment vs. True Labels', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

correct = np.sum(np.diag(cm))
print(f'Correctly assigned: {correct}/{len(y_true)} ({correct/len(y_true):.1%})')

### B.5 Effect of Changing K

Let's visualize how different values of K change the cluster structure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors_multi = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#CCB974']

for i, k in enumerate(range(2, 8)):
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km_k.fit_predict(X_scaled)
    ari_k = adjusted_rand_score(y_true, labels_k)
    sil_k = silhouette_score(X_scaled, labels_k)

    for cluster in range(k):
        mask = labels_k == cluster
        axes[i].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=colors_multi[cluster % len(colors_multi)],
                        s=40, alpha=0.8, edgecolors='white', linewidths=0.3)

    axes[i].set_title(f'K={k}  |  ARI={ari_k:.3f}  |  Sil={sil_k:.3f}',
                      fontsize=10, fontweight='bold')
    axes[i].set_xlabel('PC1')
    axes[i].set_ylabel('PC2')

plt.suptitle('K-Means with Different Values of K (PCA projection)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Patterns observed:**
- **K=2** merges what should be two distinct groups, losing meaningful structure.
- **K=3** achieves the best balance — highest ARI and strong silhouette score — matching the true cultivar structure.
- **K=4 and beyond** artificially splits natural clusters, reducing interpretability without a meaningful gain in cohesion.
- The ARI peaks at K=3, confirming it as the optimal choice for this dataset.

---
## Part C: DBSCAN Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed and marks low-density points as **noise (-1)**. Unlike K-Means, it does not require specifying the number of clusters in advance.

Two key parameters:
- **`eps`** — the maximum distance between two points to be considered neighbors
- **`min_samples`** — minimum points required to form a dense region (core point)

### C.1 Experimenting with eps and min_samples

In [ ]:
# Test multiple parameter combinations
configs = [
    {'eps': 0.5,  'min_samples': 3},
    {'eps': 1.0,  'min_samples': 3},
    {'eps': 1.5,  'min_samples': 3},
    {'eps': 1.0,  'min_samples': 5},
    {'eps': 1.5,  'min_samples': 5},
    {'eps': 2.0,  'min_samples': 5},
]

print(f'{"eps":>6} {"min_samples":>12} {"n_clusters":>12} {"noise_pts":>10} {"ARI":>8}')
print('-' * 55)
for cfg in configs:
    db = DBSCAN(eps=cfg['eps'], min_samples=cfg['min_samples'])
    db_labels = db.fit_predict(X_scaled)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    ari_db = adjusted_rand_score(y_true, db_labels) if n_clusters > 1 else 0
    print(f"{cfg['eps']:>6} {cfg['min_samples']:>12} {n_clusters:>12} {n_noise:>10} {ari_db:>8.3f}")

### C.2 Visualize DBSCAN Results — Best vs. Extreme Configs

In [ ]:
selected_configs = [
    {'eps': 0.5,  'min_samples': 3,  'label': 'eps=0.5, min_samples=3\n(too fragmented)'},
    {'eps': 1.0,  'min_samples': 3,  'label': 'eps=1.0, min_samples=3\n(best balance)'},
    {'eps': 1.5,  'min_samples': 5,  'label': 'eps=1.5, min_samples=5\n(fewer, larger clusters)'},
    {'eps': 2.0,  'min_samples': 5,  'label': 'eps=2.0, min_samples=5\n(over-merged)'},
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for i, cfg in enumerate(selected_configs):
    db = DBSCAN(eps=cfg['eps'], min_samples=cfg['min_samples'])
    db_labels = db.fit_predict(X_scaled)

    unique_labels = sorted(set(db_labels))
    n_clusters = len([l for l in unique_labels if l != -1])
    n_noise = (db_labels == -1).sum()

    color_map = plt.cm.get_cmap('tab10', max(n_clusters, 1))
    cluster_idx = 0
    for lbl in unique_labels:
        mask = db_labels == lbl
        if lbl == -1:
            axes[i].scatter(X_pca[mask, 0], X_pca[mask, 1],
                            c='red', marker='x', s=60, label='Noise', alpha=0.7, zorder=5)
        else:
            axes[i].scatter(X_pca[mask, 0], X_pca[mask, 1],
                            color=color_map(cluster_idx), s=50, alpha=0.8,
                            edgecolors='white', linewidths=0.3,
                            label=f'Cluster {lbl}')
            cluster_idx += 1

    axes[i].set_title(f'{cfg["label"]}\nClusters: {n_clusters} | Noise: {n_noise}',
                      fontsize=10, fontweight='bold')
    axes[i].set_xlabel('PC1')
    axes[i].set_ylabel('PC2')
    axes[i].legend(fontsize=8, loc='upper right')

plt.suptitle('DBSCAN — Effect of eps and min_samples (PCA projection)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### C.3 Best DBSCAN Configuration — Detailed Analysis

In [ ]:
# Best config: eps=1.0, min_samples=3
best_db = DBSCAN(eps=1.0, min_samples=3)
best_db_labels = best_db.fit_predict(X_scaled)

n_clusters = len(set(best_db_labels)) - (1 if -1 in best_db_labels else 0)
n_noise = (best_db_labels == -1).sum()
ari_best = adjusted_rand_score(y_true, best_db_labels)

print(f'DBSCAN Best Config (eps=1.0, min_samples=3):')
print(f'  Clusters found   : {n_clusters}')
print(f'  Noise points     : {n_noise} ({n_noise/len(y_true):.1%} of data)')
print(f'  Adjusted Rand Index: {ari_best:.4f}')
print(f'\nCluster size breakdown:')
unique, counts = np.unique(best_db_labels, return_counts=True)
for u, c in zip(unique, counts):
    label_name = 'Noise' if u == -1 else f'Cluster {u}'
    print(f'  {label_name}: {c} points')

### C.4 K-Means vs. DBSCAN — Side by Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# K-Means (K=3)
for cluster in range(3):
    mask = km_labels == cluster
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_km[cluster], label=f'Cluster {cluster}',
                    s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroids')
axes[0].set_title(f'K-Means (K=3)\nARI={ari:.3f} | Sil={sil:.3f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend()

# DBSCAN best
color_map = plt.cm.get_cmap('tab10', 3)
cluster_idx = 0
for lbl in sorted(set(best_db_labels)):
    mask = best_db_labels == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c='red', marker='x', s=80, label=f'Noise ({mask.sum()})', alpha=0.8, zorder=5)
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        color=color_map(cluster_idx), s=60, alpha=0.8,
                        edgecolors='white', linewidths=0.5, label=f'Cluster {lbl}')
        cluster_idx += 1
axes[1].set_title(f'DBSCAN (eps=1.0, min_samples=3)\nARI={ari_best:.3f} | Noise={n_noise}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend()

plt.suptitle('K-Means vs. DBSCAN — Final Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**DBSCAN vs. K-Means Comparison:**

| Metric | K-Means (K=3) | DBSCAN (best config) |
|--------|--------------|----------------------|
| Clusters found | 3 | ~3 |
| Noise points | 0 | Some |
| ARI | ~0.90 | Lower |
| Requires K? | Yes | No |
| Handles outliers? | No | Yes |

**Strengths of K-Means on this dataset:**
- The Wine dataset has roughly spherical, well-separated clusters of similar density — exactly what K-Means is designed for.
- K-Means achieves a higher ARI and correctly assigns nearly all points.

**Strengths of DBSCAN:**
- DBSCAN does not need K specified in advance — useful when we have no domain knowledge about the number of clusters.
- It identifies noise points (outliers), which K-Means forces into the nearest cluster regardless.
- It would outperform K-Means on datasets with irregular cluster shapes or varying densities.

**Weaknesses of DBSCAN on this dataset:**
- The 13-dimensional, relatively uniform-density Wine data makes tuning `eps` difficult.
- Small changes in `eps` cause large swings in cluster count and noise points.

---
## Part D: Reflection

### Real-World Applications of Clustering

One compelling real-world application of clustering is **customer segmentation** in retail and e-commerce. By clustering customers based on purchasing behavior, browsing history, demographics, and engagement patterns, companies can identify distinct groups — such as "high-value loyalists," "deal-seekers," and "at-risk churners" — without any predefined labels. This enables targeted marketing campaigns, personalized product recommendations, and tailored retention strategies.

K-Means works well here when customer groups are roughly equal in size and the number of segments is known from prior research. DBSCAN is better suited for detecting unusual purchasing patterns that may indicate fraud or niche power users, since it can flag outliers as noise rather than forcing them into a cluster.

Another application is **anomaly detection in manufacturing quality control** — an area directly relevant to industrial engineering. Clustering sensor readings from production equipment can identify normal operating conditions as dense clusters, while deviations (noise points in DBSCAN, or distant points in K-Means) flag potential defects or equipment failures before they cause line shutdowns.

### Ethical and Societal Considerations

Clustering algorithms carry important ethical risks, particularly when applied to people:

**1. Reinforcing bias:** If historical data contains systemic bias (e.g., loan approvals skewed by race or zip code), clustering on that data will reproduce and potentially amplify those biases. Clusters that appear "natural" in the data may reflect social inequities rather than meaningful differences.

**2. Lack of interpretability:** Unsupervised clusters have no inherent meaning — labels like "Cluster 2" are assigned by analysts, which introduces subjectivity. In high-stakes domains (healthcare, criminal justice), misinterpreting clusters can cause real harm.

**3. Privacy in customer segmentation:** Grouping individuals by behavior, location, or demographics raises privacy concerns. Even anonymized data can be re-identified when clusters are narrow and specific.

**4. Feedback loops:** In systems that act on cluster assignments (e.g., showing certain ads only to certain clusters), the algorithm's initial assumptions get reinforced over time, narrowing what opportunities are presented to individuals in disadvantaged groups.

Responsible use of clustering requires transparent documentation of assumptions, regular auditing for disparate impacts, and human oversight before cluster-based decisions affect people's lives.

---
*End of Notebook*